In [0]:
# COMMAND ----------

%pip install databricks-feature-engineering

In [0]:
# COMMAND ----------

%pip install scikit-learn mlflow

In [0]:
dbutils.library.restartPython()

In [0]:
# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

from databricks.feature_engineering import FeatureEngineeringClient
from databricks.feature_engineering import FeatureLookup

import mlflow
import mlflow.sklearn

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

print("All libraries imported successfully")

In [0]:
# COMMAND ----------

CATALOG = "retail_ai_prod"

GOLD_SCHEMA = "gold_data"
FEATURE_SCHEMA = "features"
MODEL_SCHEMA = "models"

SALES_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.sales_transactions"
)

CUSTOMER_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.customer_profiles"
)

INVENTORY_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.inventory_levels"
)

FEATURE_TABLE = (
    f"{CATALOG}.{FEATURE_SCHEMA}.customer_rfm_features"
)

MODEL_NAME = (
    f"{CATALOG}.{MODEL_SCHEMA}.customer_churn_model"
)

# Fixed snapshot date for reproducible dummy-project results
AS_OF_DATE = "2026-11-15"

print("Sales:", SALES_TABLE)
print("Customers:", CUSTOMER_TABLE)
print("Features:", FEATURE_TABLE)
print("Model:", MODEL_NAME)

In [0]:
# COMMAND ----------

fe = FeatureEngineeringClient()

print("Feature Engineering Client initialized")

In [0]:
# COMMAND ----------

sales_df = spark.table(SALES_TABLE)

customer_df = spark.table(CUSTOMER_TABLE)

inventory_df = spark.table(INVENTORY_TABLE)

print("Sales:", sales_df.count())
print("Customers:", customer_df.count())
print("Inventory:", inventory_df.count())

display(sales_df.limit(5))
display(customer_df.limit(5))

In [0]:
# COMMAND ----------

display(
    sales_df.select(
        "transaction_id",
        "transaction_date",
        "customer_id",
        "product_id",
        "quantity",
        "net_revenue"
    ).orderBy("transaction_date", ascending=False)
)

## Calculate customer-level RFM features

In [0]:
# COMMAND ----------

AS_OF_DATE = F.to_date(F.lit(AS_OF_DATE))

sales_aggregated = (
    sales_df
    .groupBy("customer_id")
    .agg(
        F.max("transaction_date").alias("last_purchase_date"),

        F.countDistinct("transaction_id").alias("frequency"),

        F.sum("net_revenue").alias("monetary"),

        F.sum("quantity").alias("total_units_purchased"),

        F.sum("gross_sales").alias("total_gross_sales"),

        F.sum("returns_amount").alias("total_returns"),

        F.countDistinct("category").alias("category_diversity"),

        F.avg("net_revenue").alias("avg_order_value")
    )
)

display(sales_aggregated)

In [0]:
# COMMAND ----------

sales_aggregated = (
    sales_aggregated
    .withColumn(
        "recency_days",
        F.datediff(
            AS_OF_DATE,
            F.col("last_purchase_date")
        )
    )
)

display(
    sales_aggregated.select(
        "customer_id",
        "last_purchase_date",
        "recency_days"
    )
)

In [0]:
# COMMAND ----------

customer_features_df = (
    customer_df.alias("c")
    .join(
        sales_aggregated.alias("s"),
        on="customer_id",
        how="left"
    )
)

display(customer_features_df)

In [0]:
# COMMAND ----------

customer_features_df = (
    customer_features_df

    .fillna({
        "frequency": 0,
        "monetary": 0.0,
        "total_units_purchased": 0,
        "total_gross_sales": 0.0,
        "total_returns": 0.0,
        "category_diversity": 0,
        "avg_order_value": 0.0
    })

    .withColumn(
        "recency_days",
        F.coalesce(
            F.col("recency_days"),
            F.lit(9999)
        )
    )
)

In [0]:
# COMMAND ----------

customer_features_df = (
    customer_features_df

    .withColumn(
        "return_rate",
        F.when(
            F.col("total_gross_sales") > 0,
            F.col("total_returns") /
            F.col("total_gross_sales")
        ).otherwise(0.0)
    )

    .withColumn(
        "customer_tenure_days",
        F.datediff(
            AS_OF_DATE,
            F.col("signup_date")
        )
    )

    .withColumn(
        "avg_order_value",
        F.when(
            F.col("frequency") > 0,
            F.col("monetary") / F.col("frequency")
        ).otherwise(0.0)
    )
)

display(customer_features_df)

In [0]:
# COMMAND ----------

recency_window = Window.orderBy(
    F.col("recency_days").asc()
)

frequency_window = Window.orderBy(
    F.col("frequency").desc()
)

monetary_window = Window.orderBy(
    F.col("monetary").desc()
)

customer_features_df = (
    customer_features_df

    .withColumn(
        "r_score",
        F.ntile(5).over(recency_window)
    )

    .withColumn(
        "f_score",
        F.ntile(5).over(frequency_window)
    )

    .withColumn(
        "m_score",
        F.ntile(5).over(monetary_window)
    )

    .withColumn(
        "rfm_score",
        F.col("r_score") +
        F.col("f_score") +
        F.col("m_score")
    )
)

In [0]:
# COMMAND ----------

customer_features_df = (
    customer_features_df
    .withColumn(
        "customer_segment",

        F.when(
            (F.col("r_score") >= 4) &
            (F.col("f_score") >= 4) &
            (F.col("m_score") >= 4),
            "Champions"
        )

        .when(
            (F.col("r_score") >= 3) &
            (F.col("f_score") >= 3),
            "Loyal Customers"
        )

        .when(
            F.col("r_score") <= 2,
            "At Risk"
        )

        .otherwise("Regular Customers")
    )
)

display(
    customer_features_df.select(
        "customer_id",
        "recency_days",
        "frequency",
        "monetary",
        "rfm_score",
        "customer_segment"
    )
)

In [0]:
# COMMAND ----------

customer_features_df = (
    customer_features_df

    .withColumn(
        "loyalty_score",
        F.when(F.col("loyalty_tier") == "Bronze", 1)
        .when(F.col("loyalty_tier") == "Silver", 2)
        .when(F.col("loyalty_tier") == "Gold", 3)
        .when(F.col("loyalty_tier") == "Platinum", 4)
        .otherwise(0)
    )

    .withColumn(
        "channel_score",
        F.when(F.col("preferred_channel") == "App", 1)
        .when(F.col("preferred_channel") == "Web", 2)
        .when(F.col("preferred_channel") == "Store", 3)
        .otherwise(0)
    )
)

In [0]:
# COMMAND ----------

customer_rfm_features = customer_features_df.select(
    "customer_id",

    "recency_days",
    "frequency",
    "monetary",
    "avg_order_value",

    "total_units_purchased",
    "total_returns",
    "return_rate",

    "category_diversity",
    "customer_tenure_days",

    "r_score",
    "f_score",
    "m_score",
    "rfm_score",

    "loyalty_score",
    "channel_score",

    "customer_segment"
)

display(customer_rfm_features)

In [0]:
# COMMAND ----------

print("Total customers:", customer_rfm_features.count())

print(
    "Unique customer IDs:",
    customer_rfm_features
    .select("customer_id")
    .distinct()
    .count()
)

print(
    "Null customer IDs:",
    customer_rfm_features
    .filter(F.col("customer_id").isNull())
    .count()
)

assert customer_rfm_features.count() == customer_df.count()

assert (
    customer_rfm_features
    .select("customer_id")
    .distinct()
    .count()
    == customer_rfm_features.count()
)

assert (
    customer_rfm_features
    .filter(F.col("customer_id").isNull())
    .count()
    == 0
)

print("Customer feature validation passed")

In [0]:
# COMMAND ----------

sales_90d = (
    sales_df
    .filter(
        F.col("transaction_date") >=
        F.date_sub(AS_OF_DATE, 90)
    )
    .groupBy("product_id", "store_city")
    .agg(
        F.sum("quantity").alias("units_sold_90d"),
        F.sum("net_revenue").alias("revenue_90d")
    )
)

display(sales_90d)

In [0]:
# COMMAND ----------

product_inventory_features = (
    inventory_df.alias("i")
    .join(
        sales_90d.alias("s"),
        on=["product_id", "store_city"],
        how="left"
    )

    .fillna({
        "units_sold_90d": 0,
        "revenue_90d": 0.0
    })

    .withColumn(
        "available_qty",
        F.col("on_hand_qty") - F.col("reserved_qty")
    )

    .withColumn(
        "avg_daily_units_sold",
        F.col("units_sold_90d") / 90.0
    )

    .withColumn(
        "stock_cover_days",
        F.when(
            F.col("avg_daily_units_sold") > 0,
            F.col("available_qty") /
            F.col("avg_daily_units_sold")
        ).otherwise(None)
    )

    .withColumn(
        "reorder_required",
        F.when(
            F.col("available_qty") < F.col("reorder_level"),
            1
        ).otherwise(0)
    )

    .select(
        "product_id",
        "store_city",
        "on_hand_qty",
        "reserved_qty",
        "available_qty",
        "reorder_level",
        "unit_cost",
        "units_sold_90d",
        "revenue_90d",
        "avg_daily_units_sold",
        "stock_cover_days",
        "reorder_required"
    )
)

display(product_inventory_features)

In [0]:
# COMMAND ----------

assert (
    product_inventory_features
    .groupBy("product_id", "store_city")
    .count()
    .filter(F.col("count") > 1)
    .count() == 0
)

assert (
    product_inventory_features
    .filter(F.col("available_qty") < 0)
    .count() == 0
)

print("Product inventory features validated")

##Create a synthetic churn label

In [0]:
# COMMAND ----------

CHURN_THRESHOLD_DAYS = 120

churn_labels_df = (
    customer_rfm_features
    .select(
        "customer_id",
        F.when(
            F.col("recency_days") >= CHURN_THRESHOLD_DAYS,
            1
        ).otherwise(0).alias("is_churn_proxy")
    )
)

display(churn_labels_df)

In [0]:
# COMMAND ----------

display(
    churn_labels_df
    .groupBy("is_churn_proxy")
    .count()
    .orderBy("is_churn_proxy")
)

##Register customer features in Unity Catalog

In [0]:
# COMMAND ----------

if not spark.catalog.tableExists(FEATURE_TABLE):

    fe.create_table(
        name=FEATURE_TABLE,
        primary_keys=["customer_id"],
        df=customer_rfm_features,
        description=(
            "Customer RFM and behavioral features "
            "for churn prediction and segmentation"
        ),
        tags={
            "project": "retail_supply_chain_copilot",
            "feature_group": "customer_rfm"
        }
    )

    print("Customer feature table created")

else:

    fe.write_table(
        name=FEATURE_TABLE,
        df=customer_rfm_features,
        mode="merge"
    )

    print("Customer feature table updated")

In [0]:
# COMMAND ----------

PRODUCT_FEATURE_TABLE = (
    f"{CATALOG}.{FEATURE_SCHEMA}.product_inventory_features"
)

if not spark.catalog.tableExists(PRODUCT_FEATURE_TABLE):

    fe.create_table(
        name=PRODUCT_FEATURE_TABLE,
        primary_keys=["product_id", "store_city"],
        df=product_inventory_features,
        description=(
            "Product and store inventory features "
            "for replenishment and demand forecasting"
        ),
        tags={
            "project": "retail_supply_chain_copilot",
            "feature_group": "inventory"
        }
    )

else:

    fe.write_table(
        name=PRODUCT_FEATURE_TABLE,
        df=product_inventory_features,
        mode="merge"
    )

print("Product inventory feature table ready")

In [0]:
# COMMAND ----------

registered_customer_features = fe.read_table(
    name=FEATURE_TABLE
)

registered_inventory_features = fe.read_table(
    name=PRODUCT_FEATURE_TABLE
)

display(registered_customer_features)
display(registered_inventory_features)

In [0]:
# COMMAND ----------

customer_feature_metadata = fe.get_table(
    name=FEATURE_TABLE
)

print(customer_feature_metadata)

In [0]:
%sql
-- COMMAND ----------

DESCRIBE TABLE EXTENDED
retail_ai_prod.features.customer_rfm_features;

In [0]:
%sql
-- COMMAND ----------

SELECT
    customer_segment,
    COUNT(*) AS customers,
    ROUND(AVG(monetary), 2) AS avg_monetary,
    ROUND(AVG(frequency), 2) AS avg_frequency
FROM retail_ai_prod.features.customer_rfm_features
GROUP BY customer_segment
ORDER BY customers DESC;

## Create a training dataset using FeatureLookup

In [0]:
# COMMAND ----------

from databricks.feature_engineering import FeatureLookup

print("FeatureLookup imported")

In [0]:
# COMMAND ----------

feature_lookups = [

    FeatureLookup(
        table_name=FEATURE_TABLE,
        lookup_key="customer_id",
        feature_name="frequency"
    ),

    FeatureLookup(
        table_name=FEATURE_TABLE,
        lookup_key="customer_id",
        feature_name="monetary"
    ),

    FeatureLookup(
        table_name=FEATURE_TABLE,
        lookup_key="customer_id",
        feature_name="avg_order_value"
    ),

    FeatureLookup(
        table_name=FEATURE_TABLE,
        lookup_key="customer_id",
        feature_name="total_units_purchased"
    ),

    FeatureLookup(
        table_name=FEATURE_TABLE,
        lookup_key="customer_id",
        feature_name="total_returns"
    ),

    FeatureLookup(
        table_name=FEATURE_TABLE,
        lookup_key="customer_id",
        feature_name="return_rate"
    ),

    FeatureLookup(
        table_name=FEATURE_TABLE,
        lookup_key="customer_id",
        feature_name="category_diversity"
    ),

    FeatureLookup(
        table_name=FEATURE_TABLE,
        lookup_key="customer_id",
        feature_name="customer_tenure_days"
    ),

    FeatureLookup(
        table_name=FEATURE_TABLE,
        lookup_key="customer_id",
        feature_name="loyalty_score"
    ),

    FeatureLookup(
        table_name=FEATURE_TABLE,
        lookup_key="customer_id",
        feature_name="channel_score"
    )
]

print("Feature lookups defined:", len(feature_lookups))

In [0]:
# COMMAND ----------

training_spine_df = churn_labels_df.select(
    "customer_id",
    "is_churn_proxy"
)

training_set = fe.create_training_set(
    df=training_spine_df,

    feature_lookups=feature_lookups,

    label="is_churn_proxy",

    exclude_columns=["customer_id"]
)

print("Training set created successfully")

In [0]:
# COMMAND ----------

training_spark_df = training_set.load_df()

display(training_spark_df)

In [0]:
# COMMAND ----------

print("Training rows:", training_spark_df.count())

print(
    "Missing labels:",
    training_spark_df
    .filter(F.col("is_churn_proxy").isNull())
    .count()
)

print(
    "Missing feature rows:",
    training_spark_df
    .filter(F.col("frequency").isNull())
    .count()
)

display(
    training_spark_df
    .groupBy("is_churn_proxy")
    .count()
)

## Train the churn prediction model

In [0]:
# COMMAND ----------

training_pdf = training_spark_df.toPandas()

print("Pandas shape:", training_pdf.shape)

display(training_pdf.head())

In [0]:
# COMMAND ----------

FEATURE_COLUMNS = [
    "frequency",
    "monetary",
    "avg_order_value",
    "total_units_purchased",
    "total_returns",
    "return_rate",
    "category_diversity",
    "customer_tenure_days",
    "loyalty_score",
    "channel_score"
]

TARGET_COLUMN = "is_churn_proxy"

X = training_pdf[FEATURE_COLUMNS].astype(float)

y = training_pdf[TARGET_COLUMN].astype(int)

print("X shape:", X.shape)
print("Y shape:", y.shape)

print("Churn distribution:")
print(y.value_counts())

In [0]:
# COMMAND ----------

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.25,

    random_state=42,

    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

In [0]:
# COMMAND ----------

churn_model = RandomForestClassifier(

    n_estimators=100,

    max_depth=5,

    min_samples_split=4,

    min_samples_leaf=2,

    class_weight="balanced",

    random_state=42
)

churn_model.fit(
    X_train,
    y_train
)

print("Churn model training completed!")

In [0]:
# COMMAND ----------

y_pred = churn_model.predict(X_test)

y_probability = churn_model.predict_proba(X_test)[:, 1]

print("Predictions completed")

In [0]:
# COMMAND ----------

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test,
    y_probability
)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("ROC AUC:", roc_auc)

In [0]:
# COMMAND ----------

print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)

print("Confusion Matrix:")

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

In [0]:
# COMMAND ----------

importance_df = pd.DataFrame({

    "feature": FEATURE_COLUMNS,

    "importance": churn_model.feature_importances_

}).sort_values(
    "importance",
    ascending=False
)

display(importance_df)

##Log the model using MLflow and Feature Engineering

In [0]:
# COMMAND ----------

assert training_pdf[FEATURE_COLUMNS].notnull().all().all(), (
    "Missing features detected. Fix the feature pipeline "
    "before model training."
)

X = training_pdf[FEATURE_COLUMNS].astype(float)
y = training_pdf[TARGET_COLUMN].astype(int)

print("Training features validated without additional transformations")

In [0]:
# COMMAND ----------

mlflow.set_registry_uri("databricks-uc")

mlflow.set_experiment(
    "/Shared/retail_customer_churn_experiment"
)

print("MLflow configured")

In [0]:
# COMMAND ----------

with mlflow.start_run(
    run_name="retail_customer_churn_rf"
) as run:

    # Log hyperparameters

    mlflow.log_params({
        "algorithm": "RandomForestClassifier",
        "n_estimators": 100,
        "max_depth": 5,
        "churn_threshold_days": CHURN_THRESHOLD_DAYS,
        "test_size": 0.25,
        "random_state": 42
    })

    # Log evaluation metrics

    mlflow.log_metrics({
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "roc_auc": float(roc_auc)
    })

    # Log model with feature metadata

    fe.log_model(
        model=churn_model,

        artifact_path="model",

        flavor=mlflow.sklearn,

        training_set=training_set,

        registered_model_name=MODEL_NAME,

        await_registration_for=300,

        infer_input_example=True
    )

    print("MLflow run ID:", run.info.run_id)

print("Model logged and registered!")

In [0]:
# COMMAND ----------

from mlflow.tracking import MlflowClient

client = MlflowClient()

model_versions = client.search_model_versions(
    f"name='{MODEL_NAME}'"
)

for version in model_versions:

    print("Model:", version.name)
    print("Version:", version.version)
    print("Status:", version.status)
    print("Run ID:", version.run_id)

##Perform inference using automatic feature lookup

In [0]:
# COMMAND ----------

model_versions = client.search_model_versions(
    f"name='{MODEL_NAME}'"
)

latest_version = max(
    model_versions,
    key=lambda x: int(x.version)
).version

MODEL_URI = f"models:/{MODEL_NAME}/{latest_version}"

print("Model URI:", MODEL_URI)

In [0]:
# COMMAND ----------

inference_df = (
    spark.table(CUSTOMER_TABLE)
    .select("customer_id")
    .limit(10)
)

display(inference_df)

In [0]:
# COMMAND ----------

predictions_df = fe.score_batch(
    model_uri=MODEL_URI,
    df=inference_df
)

display(predictions_df)

In [0]:
# COMMAND ----------

business_predictions = (
    predictions_df
    .withColumn(
        "churn_status",
        F.when(
            F.col("prediction") == 1,
            "Potentially Inactive"
        ).otherwise("Active")
    )
    .select(
        "customer_id",
        "prediction",
        "churn_status"
    )
)

display(business_predictions)

##Create a Feature Serving Endpoint

In [0]:
# COMMAND ----------

from databricks.feature_engineering import (
    FeatureEngineeringClient,
    FeatureLookup
)

FEATURE_SPEC_NAME = (
    "retail_ai_prod.features.customer_rfm_serving_spec"
)

serving_features = [

    FeatureLookup(
        table_name=FEATURE_TABLE,

        lookup_key="customer_id",

        feature_names=[
            "frequency",
            "monetary",
            "avg_order_value",
            "total_units_purchased",
            "total_returns",
            "return_rate",
            "category_diversity",
            "customer_tenure_days",
            "loyalty_score",
            "channel_score"
        ]
    )
]

try:

    fe.create_feature_spec(
        name=FEATURE_SPEC_NAME,
        features=serving_features
    )

    print("FeatureSpec created")

except Exception as e:

    if "already exists" in str(e).lower():

        print("FeatureSpec already exists")

    else:

        raise

In [0]:
# COMMAND ----------

from databricks.feature_engineering.entities.feature_serving_endpoint import (
    ServedEntity,
    EndpointCoreConfig
)

FEATURE_ENDPOINT_NAME = "retail-customer-features"

endpoint_config = EndpointCoreConfig(

    served_entities=ServedEntity(

        feature_spec_name=FEATURE_SPEC_NAME,

        workload_size="Small",

        scale_to_zero_enabled=True
    )
)

feature_endpoint = fe.create_feature_serving_endpoint(

    name=FEATURE_ENDPOINT_NAME,

    config=endpoint_config
)

print("Feature Serving endpoint creation submitted")
print(feature_endpoint)

In [0]:
import inspect
import databricks.feature_engineering as dfe
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()
print("FE package version:", getattr(dfe, "__version__", "Not exposed"))

for name in [
    "list_online_stores",
    "get_online_store",
    "create_online_store",
    "publish_table",
    "get_feature_spec",
    "get_feature_serving_endpoint",
    "create_feature_serving_endpoint",
    "update_feature_serving_endpoint",
]:
    present = hasattr(fe, name)
    sig = None
    if present:
        try:
            sig = str(inspect.signature(getattr(fe, name)))
        except Exception as e:
            sig = f"<signature error: {e}>"
    print(f"{name}: present={present} | sig={sig}")

print("\n--- Offline Table Detail ---")
display(spark.sql("DESCRIBE DETAIL retail_ai_prod.features.customer_rfm_features"))

print("\n--- Catalog Table Checks ---")
for tbl in [
    "retail_ai_prod.features.customer_rfm_features",
    "retail_ai_prod.features.customer_rfm_features_online",
    "retail_ai_prod.features.product_inventory_features",
]:
    print(f"{tbl} exists:", spark.catalog.tableExists(tbl))

print("\n--- Online Stores ---")
try:
    print("list_online_stores():", fe.list_online_stores())
except Exception as e:
    print("list_online_stores() error:", repr(e))

print("\n--- Feature Spec Check ---")
try:
    if hasattr(fe, "get_feature_spec"):
        print(fe.get_feature_spec(name="retail_ai_prod.features.customer_rfm_serving_spec"))
    else:
        print("get_feature_spec not on client; checking via SQL DESCRIBE EXTENDED...")
        display(spark.sql("DESCRIBE EXTENDED retail_ai_prod.features.customer_rfm_serving_spec"))
except Exception as e:
    print("Feature spec check error:", repr(e))

print("\n--- Endpoint State & Config ---")
try:
    ep = fe.get_feature_serving_endpoint(name="retail-customer-features")
    print("state:", ep.state)
    print("ready:", ep.state.get("ready") if isinstance(ep.state, dict) else None)
    print("config_update:", ep.state.get("config_update") if isinstance(ep.state, dict) else None)
    print("config:", getattr(ep, "config", ep))
except Exception as e:
    print("Endpoint check error:", repr(e))

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

print("=== 1. Offline Table Metadata (SQL) ===")
desc_df = spark.sql("DESCRIBE EXTENDED retail_ai_prod.features.customer_rfm_features")
desc_df.show(truncate=False)

detail_df = spark.sql("DESCRIBE DETAIL retail_ai_prod.features.customer_rfm_features")
detail_df.select("id", "name", "format", "location", "properties").show(truncate=False)

print("\n=== 2. Online Stores List ===")
try:
    stores = fe.list_online_stores()
    print("Online stores count:", len(stores))
    for s in stores:
        print("Store:", s)
except Exception as e:
    print("list_online_stores() failed:", repr(e))

print("\n=== 3. Feature Serving Endpoint State ===")
try:
    ep = fe.get_feature_serving_endpoint(name="retail-customer-features")
    print("Endpoint name:", getattr(ep, "name", None))
    print("Endpoint state:", getattr(ep, "state", None))
    if isinstance(getattr(ep, "state", None), dict):
        print("ready:", ep.state.get("ready"))
        print("config_update:", ep.state.get("config_update"))
except Exception as e:
    print("get_feature_serving_endpoint() failed:", repr(e))

## Create a product inventory feature lookup

In [0]:
# COMMAND ----------
# Poll until the Feature Serving endpoint is READY
import time

MAX_WAIT_SECONDS = 600   # 10 minutes
POLL_INTERVAL   = 20

start = time.time()
while True:
    endpoint = fe.get_feature_serving_endpoint(name=FEATURE_ENDPOINT_NAME)
    ready    = endpoint.state.get("ready", "")
    cfg_upd  = endpoint.state.get("config_update", "")

    print(f"[{int(time.time()-start)}s] ready={ready}  config_update={cfg_upd}")

    if ready == "READY":
        print("\nEndpoint is READY.")
        break

    if cfg_upd == "UPDATE_FAILED":
        raise RuntimeError(
            f"Endpoint update FAILED. Full state: {endpoint.state}"
        )

    if time.time() - start > MAX_WAIT_SECONDS:
        raise TimeoutError(
            f"Endpoint not ready after {MAX_WAIT_SECONDS}s. State: {endpoint.state}"
        )

    time.sleep(POLL_INTERVAL)

## Product Inventory Feature Lookup

In [0]:
# COMMAND ----------
# Read registered inventory features from Unity Catalog Feature Store
PRODUCT_FEATURE_TABLE = (
    f"{CATALOG}.{FEATURE_SCHEMA}.product_inventory_features"
)

inventory_features_df = fe.read_table(name=PRODUCT_FEATURE_TABLE)

print("Inventory feature rows:", inventory_features_df.count())
print("Columns:", inventory_features_df.columns)
display(inventory_features_df.limit(10))

In [0]:
# COMMAND ----------
# Products that have fallen below reorder level
reorder_alerts = (
    inventory_features_df
    .filter(F.col("reorder_required") == True)
    .select(
        "product_id",
        "store_city",
        "on_hand_qty",
        "reorder_level",
        "stock_cover_days",
        "avg_daily_units_sold"
    )
    .orderBy("stock_cover_days")   # most urgent first
)

print(f"Products requiring reorder: {reorder_alerts.count()}")
display(reorder_alerts)

In [0]:
# COMMAND ----------
# Stock health summary: how many products are healthy vs low vs critical
stock_health = (
    inventory_features_df
    .withColumn(
        "stock_status",
        F.when(F.col("stock_cover_days") >= 30, "Healthy")
         .when(F.col("stock_cover_days") >= 7,  "Low")
         .otherwise("Critical")
    )
    .groupBy("store_city", "stock_status")
    .agg(F.count("*").alias("product_count"))
    .orderBy("store_city", "stock_status")
)

display(stock_health)

## Summary — What Notebook 03 Delivers

In [0]:
# COMMAND ----------
# Quick summary of everything this notebook produced

total_customers   = customer_rfm_features.count()
churned_customers = customer_rfm_features.filter(F.col("recency_days") >= 120).count()
churn_rate        = round(churned_customers / total_customers * 100, 1)

total_products    = inventory_features_df.count()
reorder_count     = inventory_features_df.filter(F.col("reorder_required") == True).count()

print("=" * 55)
print("  RETAIL AI COPILOT — NOTEBOOK 03 SUMMARY")
print("=" * 55)
print(f"  Customers analysed          : {total_customers}")
print(f"  Estimated churn rate        : {churn_rate}%  ({churned_customers} customers)")
print(f"  Inventory SKU-locations     : {total_products}")
print(f"  Products requiring reorder  : {reorder_count}")
print()
print("  Outputs registered in Unity Catalog:")
print(f"  • {FEATURE_TABLE}")
print(f"  • {PRODUCT_FEATURE_TABLE}")
print(f"  • {MODEL_NAME}")
print(f"  • Feature Spec  : {FEATURE_SPEC_NAME}")
print(f"  • Serving endpoint: {FEATURE_ENDPOINT_NAME}")
print("=" * 55)